# Embedding API (preview)

<a target="_blank" href="https://colab.research.google.com/github/neo4j/graph-data-science-client/blob/main/examples/embedding-api.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

This Jupyter notebook is hosted [here](https://github.com/neo4j/graph-data-science-client/blob/main/examples/embedding-api.ipynb) in the Neo4j Graph Data Science Client Github repository.

\[embeddings description\]

This notebook will show \[...\] on a graph dataset.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("sessions.env")

True

In [2]:
from graphdatascience.session import GdsSessions, AuraAPICredentials, SessionMemory, CloudLocation

CLIENT_ID = os.environ.get("CLIENT_ID")
CLIENT_SECRET = os.environ.get("CLIENT_SECRET")
PROJECT_ID = None

# Create a new GdsSessions object
sessions = GdsSessions(api_credentials=AuraAPICredentials(CLIENT_ID, CLIENT_SECRET, PROJECT_ID))

In [3]:
!uv pip install scipy torch torch-geometric pandas

Using Python 3.12.0 environment at: /Users/alfred/graph-data-science-client/.venv
Audited 4 packages in 22ms


In [4]:
from torch_geometric.datasets import Planetoid
import pandas as pd

dataset = Planetoid(root="datasets", name="Cora")

/Users/alfred/graph-data-science-client/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [5]:
n = dataset.x.shape[0]
m = dataset.edge_index.shape[0]
nodes_df = pd.DataFrame({"nodeId": range(n), "x": dataset.x.tolist(), "y": dataset.y, "labels": ["Paper"]*n})
# nodes_df = pd.DataFrame({"nodeId": range(n), "x": [[0., 0., 1.] for _ in range(n)], "y": dataset.y, "labels": ["Paper"]*n})
rels_df = pd.DataFrame({"sourceNodeId": dataset.edge_index[0], "targetNodeId": dataset.edge_index[1], "relationshipType": "CITES"})

In [6]:
from graphdatascience.session import SessionMemory, CloudLocation
gds = sessions.get_or_create(
    session_name="my_session_123",
    memory=SessionMemory.m_2GB,
    cloud_location=CloudLocation(provider="gcp", region="europe-west1")
)

In [12]:
gds.graph.drop("cora", fail_if_missing=False)
G = gds.graph.construct(graph_name="cora", nodes=nodes_df, relationships=rels_df)
print(G)
print(G.node_properties())
print(G.relationship_properties())

Constructing graph:   0%|          | 0.0/100 [00:00<?, ?%/s]

Graph(name=cora, node_count=2708, relationship_count=10556)
{'Paper': ['x', 'y']}
{'CITES': []}


In [11]:
from graphdatascience.procedure_surface.api.node_embedding.config import FastRPConfig

fastrp_result = gds.embeddings.encode.stream(
    G=G,
    graph_encoder=FastRPConfig(),
)
fastrp_result

/Users/alfred/graph-data-science-client/src/graphdatascience/procedure_surface/arrow/node_embedding/embeddings_arrow_endpoints.py:32: UserWarning: embeddings.encode is a preview feature and may change or be removed in future releases.
  return EncodeArrowEndpoints(self._arrow_client, self._write_protocol, show_progress=self._show_progress)


,nodeId,embeddings
0,0,[ 4.67302084e-01 2.00582668e-02 1.39261782e-...
1,1,[-0.12157682 0.01266266 0.19324106 -0.225065...
2,2,[-0.13497533 -0.08965699 0.09716658 -0.099011...
3,3,[ 0. 0. 0. 0.158113...
4,4,[ 0.08378547 0.40176278 -0.01977956 -0.112653...
...,...,...
2703,2703,[ 0. 0. 0.1490712 0.149071...
2704,2704,[-0.14285713 -0.31622773 0. -0.142857...
2705,2705,[-0.46322858 0. 0. 0. ...
2706,2706,[ 0.48439664 -0.0558343 -0.3852444 -0.485845...


In [232]:
from graphdatascience.procedure_surface.api.node_embedding.config import GraphSAGEConfig, MLPClassifierConfig

result = gds.embeddings.train.__call__(
    G=G,
    graph_encoder=GraphSAGEConfig(target_type="Paper", out_dim=1),
    decoder=MLPClassifierConfig(),
    model_save_name="cora_model_1",
    # target = ("Paper", "y"),
    target_label="Paper",
    target_property="y",
    feature_properties=["x"]
)
result

/Users/alfred/graph-data-science-client/src/graphdatascience/procedure_surface/arrow/node_embedding/embeddings_arrow_endpoints.py:24: UserWarning: embeddings.train is a preview feature and may change or be removed in future releases.
  return TrainArrowEndpoints(self._arrow_client, self._write_protocol, show_progress=self._show_progress)


FlightCancelledError: Flight cancelled call, with message: Arrow process 'GML_TRAIN' was aborted: Remote job was aborted by the runtime, reason: Failed to run task: Cannot interpret 'list<x.inner: double not null>[pyarrow]' as a data type
Remote job was aborted by the runtime, reason: Failed to run task: Cannot interpret 'list<x.inner: double not null>[pyarrow]' as a data type

In [221]:
from graphdatascience.procedure_surface.api.node_embedding.config import GraphSAGEConfig, MLPClassifierConfig

result = gds.embeddings.train.__call__(
    G=G,
    graph_encoder=GraphSAGEConfig(target_type="Paper", out_dim=1),
    decoder=MLPClassifierConfig(),
    model_save_name="cora_model_1",
    # target = ("Paper", "y"),
    target_label="Paper",
    target_property="y",
    feature_properties=["x"]
)
result

/Users/alfred/graph-data-science-client/src/graphdatascience/procedure_surface/arrow/node_embedding/embeddings_arrow_endpoints.py:24: UserWarning: embeddings.train is a preview feature and may change or be removed in future releases.
  return TrainArrowEndpoints(self._arrow_client, self._write_protocol, show_progress=self._show_progress)


FlightCancelledError: Flight cancelled call, with message: Arrow process 'GML_TRAIN' was aborted: Remote job was aborted by the runtime, reason: Failed to run task: Cannot interpret 'list<x.inner: double not null>[pyarrow]' as a data type
Remote job was aborted by the runtime, reason: Failed to run task: Cannot interpret 'list<x.inner: double not null>[pyarrow]' as a data type

In [186]:
from graphdatascience.procedure_surface.api.node_embedding.config import IdentityConfig, GBClassifierConfig

result = gds.embeddings.train.__call__(
    G=G,
    graph_encoder=IdentityConfig(target_type="Paper", out_dim=1433),
    decoder=GBClassifierConfig(),
    model_save_name="cora_model_2",
    # target = ("Paper", "y"),
    target_label="Paper",
    target_property="y",
    feature_properties=["x"]
)
result

/Users/alfred/graph-data-science-client/src/graphdatascience/procedure_surface/arrow/node_embedding/embeddings_arrow_endpoints.py:24: UserWarning: embeddings.train is a preview feature and may change or be removed in future releases.
  return TrainArrowEndpoints(self._arrow_client, self._write_protocol, show_progress=self._show_progress)


FlightCancelledError: Flight cancelled call, with message: Arrow process 'GML_TRAIN' was aborted: Remote job was aborted by the runtime, reason: Failed to run task: 'Paper'
Remote job was aborted by the runtime, reason: Failed to run task: 'Paper'

In [ ]:
predict_df = gds.embeddings.predict.stream(
    G=G,
    model_name="cora_model_1",
    feature_properties=["x"],
)
predict_df

In [ ]:
embeddings_df = gds.embeddings.encode.stream(
    G=graph,
    graph_encoder="my_new_model1234",
    feature_properties=["x"]
)
embeddings_df

In [ ]:
df